In [ ]:
# Install the packages this lab needs:
#   openai          -> OpenAI-compatible client used to call OpenRouter
#   python-dotenv   -> loads the API key from a local .env file
%pip install -q openai python-dotenv



[notice] A new release of pip available: 22.3 -> 26.2
[notice] To update, run: /Users/ayushsingh/Work/Labs/SDK/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Imports. Each tool defined below relies on these modules.
import os                 # environment variables (the API key)
import json               # serializes tool-call arguments and results
import re                 # regex engine powering the grep tool
from pathlib import Path  # filesystem paths for the read_file / glob tools
from dotenv import load_dotenv  # loads secrets from .env
from openai import OpenAI       # OpenAI-compatible client (pointed at OpenRouter)


In [ ]:
# Load OPENROUTER_API_KEY from the .env file (searched upward from this notebook's folder).
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
print(f"API key loaded: {'Yes' if OPENROUTER_API_KEY else 'No'}")


API key loaded: Yes


In [ ]:
# Tool 1 of 3: read_file
# A tool has two parts:
#   1) the Python function -> executed locally when the model asks for it
#   2) the JSON schema     -> what the model actually sees (name, params, description)
def read_file(path):
    """Read a file and return its contents."""
    return Path(path).read_text()

read_file_schema = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "Read the contents of a file.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path to the file"}
            },
            "required": ["path"]
        }
    }
}


In [ ]:
# Tool 2 of 3: glob
# Helper that makes recursive patterns like data/**/* also match top-level files.
def _expand_glob(pattern):
    """Expand glob pattern, ensuring recursive wildcards like data/**/* match both top-level and nested files."""
    pattern = pattern.replace('\\', '/')
    matching_paths = list(Path('.').glob(pattern))
    if '/**/*' in pattern:
        matching_paths.extend(Path('.').glob(pattern.replace('/**/*', '/*')))
    elif pattern.endswith('/**'):
        matching_paths.extend(Path('.').glob(pattern[:-3] + '/*'))
    
    seen = set()
    unique_files = []
    for p in matching_paths:
        p_str = str(p)
        if p.is_file() and p_str not in seen:
            seen.add(p_str)
            unique_files.append(p)
    return unique_files

def glob_files(pattern):
    """Find files matching a glob pattern."""
    return [str(p) for p in _expand_glob(pattern)]

# Schema tells the model what this tool does and how to call it.
glob_schema = {
    "type": "function",
    "function": {
        "name": "glob",
        "description": "Find files matching a glob pattern like **/*.py or data/**/*.py",
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Glob pattern"}
            },
            "required": ["pattern"]
        }
    }
}


In [ ]:
# Tool 3 of 3: grep
def grep_files(pattern, path):
    """Search file contents with regex. Returns list of (file, line_num, line) tuples."""
    regex = re.compile(pattern)
    results = []
    for f in _expand_glob(path):
        try:
            for i, line in enumerate(f.read_text(encoding='utf-8', errors='ignore').splitlines(), 1):
                if regex.search(line):
                    results.append((str(f), i, line.strip()))
        except (UnicodeDecodeError, PermissionError):
            # Skip files that can't be decoded or read (e.g. binaries).
            pass
    return results

grep_schema = {
    "type": "function",
    "function": {
        "name": "grep",
        "description": "Search file contents with regex. Returns matching lines with file paths and line numbers.",
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Regex pattern to search for"},
                "path": {"type": "string", "description": "Glob pattern for files to search (e.g. **/*.py)"}
            },
            "required": ["pattern", "path"]
        }
    }
}


In [ ]:
# Register the tools in two structures:
#   TOOLS    -> list of schemas sent to the model on every API call
#   TOOL_MAP -> dict mapping tool names to Python functions for local dispatch
TOOLS = [read_file_schema, glob_schema, grep_schema]

TOOL_MAP = {
    "read_file": read_file,
    "glob": glob_files,
    "grep": grep_files,
}

print(f"Tools registered: {list(TOOL_MAP.keys())}")


Tools registered: ['read_file', 'glob', 'grep']


In [ ]:
# Create the OpenAI-compatible client pointed at OpenRouter's API.
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Free-tier model via OpenRouter; TARGET_DIR is the codebase the agent will scan.
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
TARGET_DIR = "data"

print(f"Model: {MODEL}")
print(f"Target: {TARGET_DIR}")


Model: nvidia/nemotron-3-ultra-550b-a55b:free
Target: data


In [ ]:
# The task tells the model WHAT to do, not HOW.
# The model decides the tool sequence on its own.
TASK = f"""
Scan the codebase at {TARGET_DIR} and find all TODO and FIXME comments.

For each match, report:
- File path
- Line number
- The comment text
- A brief note on what the comment is about

Organize the results as a markdown summary grouped by file.
"""


In [ ]:
# Seed the conversation: a system prompt (the agent's role) + the user task.
messages = [
    {"role": "system", "content": "You are a code exploration assistant. Scan codebases, find patterns, and produce structured markdown reports. Be thorough but concise. Always cite file paths and line numbers."},
    {"role": "user", "content": TASK},
]

tool_call_count = 0
MAX_ITERATIONS = 15   # safety cap: prevents an infinite tool-calling loop
final_answer = ""

for i in range(MAX_ITERATIONS):
    # Send the full history + tool schemas to the model.
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
    )

    if not response.choices:
        print(f"API returned no choices. Response: {response}")
        break

    choice = response.choices[0]
    message = choice.message

    if message.tool_calls:
        # The model wants to call tools: keep its message, run each call, then loop.
        messages.append(message)

        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            tool_call_count += 1

            print(f"  [Call {tool_call_count}] {func_name}({func_args})")

            # Dispatch to the matching local Python function.
            if func_name in TOOL_MAP:
                result = TOOL_MAP[func_name](**func_args)
            else:
                result = f"Unknown tool: {func_name}"

            result_str = json.dumps(result, default=str)

            # Feed the result back to the model as a 'tool'-role message.
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result_str,
            })
    else:
        # No tool calls -> the model produced its final text answer.
        # Guard against empty/None content so downstream cells never crash.
        final_answer = message.content or ""
        break

print(f"\nDone. Tool calls made: {tool_call_count}")


  [Call 1] grep({'pattern': 'TODO|FIXME', 'path': 'data/**/*'})


  [Call 2] read_file({'path': 'data/app.py'})
  [Call 3] read_file({'path': 'data/auth/jwt.py'})
  [Call 4] read_file({'path': 'data/utils/db.py'})



Done. Tool calls made: 4


In [ ]:
# Display the agent's final markdown report.
print(f"{'='*50}")
print("AGENT RESPONSE:")
print(f"{'='*50}\n")
print(final_answer)


AGENT RESPONSE:

# TODO and FIXME Comments Summary

## `data/app.py`

| Line | Type | Comment | Context |
|------|------|---------|---------|
| 6 | TODO | `Add SSL certificate configuration for HTTPS support` | In `start_server()` function - missing HTTPS/SSL setup for production deployment |
| 10 | FIXME | `Sanitize incoming request headers to prevent header injection vulnerabilities` | In `handle_request()` function - security issue with unsanitized headers |

---

## `data/auth/jwt.py`

| Line | Type | Comment | Context |
|------|------|---------|---------|
| 6 | TODO | `Implement token expiration timeout (currently tokens never expire)` | In `generate_token()` function - tokens lack expiration, security risk |
| 10 | FIXME | `Replace mock signature verification with actual HMAC-SHA256 validation` | In `verify_token()` function - using mock verification instead of real crypto validation |

---

## `data/utils/db.py`

| Line | Type | Comment | Context |
|------|------|---------|-----

In [ ]:
# Post-process the free-form report with regex to pull out quick metrics:
# TODO mentions, FIXME mentions, and unique .py files cited.
todo_count = len(re.findall(r'(?i)TODO', final_answer))
fixme_count = len(re.findall(r'(?i)FIXME', final_answer))

file_mentions = len(set(re.findall(r'\b[\w/]+\.py\b', final_answer)))

print("="*50)
print("RESULT SUMMARY")
print("="*50)
print(f"  TODO mentions:    {todo_count}")
print(f"  FIXME mentions:   {fixme_count}")
print(f"  Unique files:     {file_mentions}")
print("="*50)


RESULT SUMMARY
  TODO mentions:    5
  FIXME mentions:   5
  Unique files:     5


In [ ]:
# Second LLM call: an automated judge grades the agent's output on 4 criteria.
# It must return a compact JSON object so scores stay machine-parseable.
judge_prompt = f"""
You are an evaluation judge. Analyze the following agent output and the codebase.

AGENT OUTPUT:
{final_answer}

TASK: Find all TODO and FIXME comments in the codebase.

Evaluate on these criteria:
1. COVERAGE: Did the agent find all the TODO/FIXME comments?
2. ACCURACY: Are all reported items real TODO/FIXME comments (not false positives from strings)?
3. COMPLETENESS: Did it include file paths and line numbers?
4. FORMAT: Is the output well-organized and readable?

Score each criterion 1-5 and give an overall score. Be strict.

Respond with ONLY a compact JSON object and nothing else, using exactly these keys:
{{"coverage": <int 1-5>, "accuracy": <int 1-5>, "completeness": <int 1-5>, "format": <int 1-5>, "overall": <int 1-5>}}
"""


def extract_json_scores(text):
    """Parse a JSON object from the judge response, tolerating prose or fences."""
    if not text:
        return None
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
        return json.loads(match.group(0)) if match else None


# Free models sometimes answer with prose instead of JSON, so we retry up to 3
# times, appending a corrective re-prompt after each unparseable attempt.
judge_messages = [{"role": "user", "content": judge_prompt}]
judge_scores = None
judge_content = ""

for attempt in range(1, 4):
    judge_response = client.chat.completions.create(model=MODEL, messages=judge_messages)
    if judge_response.choices:
        judge_content = judge_response.choices[0].message.content or ""
    else:
        judge_content = ""
    judge_scores = extract_json_scores(judge_content)
    if judge_scores:
        break
    judge_messages.append({"role": "assistant", "content": judge_content})
    judge_messages.append({
        "role": "user",
        "content": "Your previous response was not valid JSON. Reply with ONLY a compact JSON object and nothing else, using exactly these keys: {\"coverage\": <int 1-5>, \"accuracy\": <int 1-5>, \"completeness\": <int 1-5>, \"format\": <int 1-5>, \"overall\": <int 1-5>}",
    })

print("=" * 50)
print("LLM JUDGE EVALUATION")
print("=" * 50)
if judge_scores:
    print(json.dumps(judge_scores, indent=2))
else:
    print(f"Judge did not return valid JSON after retries:\n{judge_content}")

LLM JUDGE EVALUATION
{
  "coverage": 3,
  "accuracy": 3,
  "completeness": 5,
  "format": 5,
  "overall": 4
}
